In [3]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {"language": "markdown"},
      "source": [
        "# F1 Race Strategy ML Pipeline (Serverless AWS)\n",
        "\n",
        "End-to-end notebook to collect FastF1 telemetry, train lap time / tire degradation / pit strategy models, and deploy them to an AWS serverless inference stack with Lambda, API Gateway, DynamoDB, and S3."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# 1. Set Up F1 ML Environment and Dependencies\n",
        "!pip install --quiet fastf1 pandas numpy scikit-learn xgboost boto3 awscli aws-xray-sdk graphviz httpx matplotlib\n",
        "\n",
        "import os, pathlib, datetime, json, math, random\n",
        "import boto3\n",
        "\n",
        "AWS_REGION = os.environ.get('AWS_REGION', 'eu-west-1')\n",
        "AWS_PROFILE = os.environ.get('AWS_PROFILE', 'default')\n",
        "F1_MODELS_BUCKET = os.environ.get('F1_MODELS_BUCKET', 'f1-ml-models')\n",
        "F1_FEATURES_TABLE = os.environ.get('F1_FEATURES_TABLE', 'f1-features')\n",
        "\n",
        "session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)\n",
        "s3 = session.client('s3')\n",
        "dynamodb = session.resource('dynamodb')\n",
        "\n",
        "print('Using region:', AWS_REGION)\n",
        "print('Models bucket:', F1_MODELS_BUCKET)\n",
        "print('Features table:', F1_FEATURES_TABLE)\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# 2. Collect F1 Telemetry with FastF1 for 2023–2024\n",
        "import fastf1\n",
        "import pandas as pd\n",
        "from pathlib import Path\n",
        "\n",
        "DATA_DIR = Path('data')\n",
        "RAW_DIR = DATA_DIR / 'raw'\n",
        "RAW_DIR.mkdir(parents=True, exist_ok=True)\n",
        "\n",
        "fastf1.Cache.enable_cache(str(RAW_DIR / 'fastf1_cache'))\n",
        "\n",
        "def collect_laps_for_season(year, races):\n",
        "    frames = []\n",
        "    for race in races:\n",
        "        print(f'Loading {year} {race} ...')\n",
        "        session = fastf1.get_session(year, race, 'R')\n",
        "        session.load()\n",
        "        laps = session.laps.reset_index(drop=True)\n",
        "        df = pd.DataFrame({\n",
        "            'year': year,\n",
        "            'grand_prix': race,\n",
        "            'driver': laps['Driver'],\n",
        "            'lap_number': laps['LapNumber'],\n",
        "            'lap_time_seconds': laps['LapTime'].dt.total_seconds(),\n",
        "            'sector1_time': laps['Sector1Time'].dt.total_seconds(),\n",
        "            'sector2_time': laps['Sector2Time'].dt.total_seconds(),\n",
        "            'sector3_time': laps['Sector3Time'].dt.total_seconds(),\n",
        "            'compound': laps['Compound'],\n",
        "            'tire_life_laps': laps['TyreLife'],\n",
        "            'track_status': laps['TrackStatus'],\n",
        "        })\n",
        "        frames.append(df)\n",
        "    return pd.concat(frames, ignore_index=True)\n",
        "\n",
        "races_2023 = ['Bahrain', 'Saudi Arabia', 'Australia', 'Azerbaijan', 'Miami']\n",
        "races_2024 = ['Bahrain', 'Saudi Arabia', 'Australia', 'Japan']\n",
        "\n",
        "laps_2023 = collect_laps_for_season(2023, races_2023)\n",
        "laps_2024 = collect_laps_for_season(2024, races_2024)\n",
        "all_laps = pd.concat([laps_2023, laps_2024], ignore_index=True)\n",
        "all_laps.to_parquet(DATA_DIR / 'processed' / 'laps_raw_from_notebook.parquet', index=False)\n",
        "all_laps.head()\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# 3. Feature Engineering for Lap Time and Tire Degradation\n",
        "import numpy as np\n",
        "\n",
        "PROCESSED_DIR = DATA_DIR / 'processed'\n",
        "PROCESSED_DIR.mkdir(parents=True, exist_ok=True)\n",
        "\n",
        "df = all_laps.copy()\n",
        "df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['lap_time_seconds'])\n",
        "\n",
        "df['fuel_load_proxy'] = (df['lap_number'].max() - df['lap_number']) / df['lap_number'].max()\n",
        "df['track_id'] = df['grand_prix'].str.lower().str.replace(' ', '_')\n",
        "compound_dummies = pd.get_dummies(df['compound'], prefix='compound')\n",
        "track_dummies = pd.get_dummies(df['track_id'], prefix='track')\n",
        "df = pd.concat([df, compound_dummies, track_dummies], axis=1)\n",
        "\n",
        "df['tire_degradation_feature'] = df['tire_life_laps'].fillna(0) * df['fuel_load_proxy'].fillna(0)\n",
        "\n",
        "base_pace = df[df['tire_life_laps'] <= 3]['lap_time_seconds'].median()\n",
        "df['lap_time_delta_vs_fresh'] = df['lap_time_seconds'] - base_pace\n",
        "\n",
        "df.to_parquet(PROCESSED_DIR / 'laps_features_from_notebook.parquet', index=False)\n",
        "df[['lap_time_seconds','tire_life_laps','fuel_load_proxy','lap_time_delta_vs_fresh']].describe()\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# 4. Train Lap Time Prediction Model and Persist to S3\n",
        "from sklearn.model_selection import train_test_split\n",
        "from sklearn.metrics import mean_squared_error, mean_absolute_error\n",
        "from sklearn.ensemble import RandomForestRegressor\n",
        "\n",
        "feature_cols = ['tire_life_laps', 'fuel_load_proxy', 'tire_degradation_feature']\n",
        "X = df[feature_cols].values.astype('float32')\n",
        "y = df['lap_time_seconds'].values.astype('float32')\n",
        "\n",
        "X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)\n",
        "\n",
        "rf = RandomForestRegressor(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42)\n",
        "rf.fit(X_train, y_train)\n",
        "\n",
        "preds = rf.predict(X_valid)\n",
        "rmse = float(math.sqrt(mean_squared_error(y_valid, preds)))\n",
        "mae = float(mean_absolute_error(y_valid, preds))\n",
        "rmse, mae\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# Persist lap time model + metadata to S3\n",
        "import io, pickle\n",
        "\n",
        "version = 'v2_2024_season'\n",
        "model_key = f'laptime-models/{version}/model.pkl'\n",
        "meta_key = f'laptime-models/{version}/metadata.pkl'\n",
        "\n",
        "meta = {\n",
        "    'version': version,\n",
        "    'training_date': datetime.datetime.utcnow().isoformat(),\n",
        "    'seasons': ['2023','2024'],\n",
        "    'feature_schema': feature_cols,\n",
        "    'rmse_seconds': rmse,\n",
        "    'mae_seconds': mae,\n",
        "}\n",
        "buf = io.BytesIO()\n",
        "pickle.dump(rf, buf)\n",
        "buf.seek(0)\n",
        "s3.upload_fileobj(buf, F1_MODELS_BUCKET, model_key)\n",
        "\n",
        "buf_meta = io.BytesIO()\n",
        "pickle.dump(meta, buf_meta)\n",
        "buf_meta.seek(0)\n",
        "s3.upload_fileobj(buf_meta, F1_MODELS_BUCKET, meta_key)\n",
        "\n",
        "print('Uploaded lap time model to s3://%s/%s' % (F1_MODELS_BUCKET, model_key))\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# 5. Train Tire Degradation Model and Persist to S3\n",
        "deg_feature_cols = ['tire_life_laps', 'fuel_load_proxy', 'tire_degradation_feature']\n",
        "X_deg = df[deg_feature_cols].values.astype('float32')\n",
        "y_deg = df['lap_time_delta_vs_fresh'].values.astype('float32')\n",
        "\n",
        "X_deg_tr, X_deg_va, y_deg_tr, y_deg_va = train_test_split(X_deg, y_deg, test_size=0.2, random_state=42)\n",
        "deg_model = RandomForestRegressor(n_estimators=200, max_depth=10, n_jobs=-1, random_state=42)\n",
        "deg_model.fit(X_deg_tr, y_deg_tr)\n",
        "deg_preds = deg_model.predict(X_deg_va)\n",
        "deg_rmse = float(math.sqrt(mean_squared_error(y_deg_va, deg_preds)))\n",
        "deg_mae = float(mean_absolute_error(y_deg_va, deg_preds))\n",
        "deg_rmse, deg_mae\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# Upload degradation model\n",
        "version_deg = 'v1_degradation'\n",
        "model_key_deg = f'degradation-models/{version_deg}/model.pkl'\n",
        "meta_key_deg = f'degradation-models/{version_deg}/metadata.pkl'\n",
        "meta_deg = {\n",
        "    'version': version_deg,\n",
        "    'training_date': datetime.datetime.utcnow().isoformat(),\n",
        "    'rmse_seconds': deg_rmse,\n",
        "    'mae_seconds': deg_mae,\n",
        "}\n",
        "buf = io.BytesIO()\n",
        "pickle.dump(deg_model, buf)\n",
        "buf.seek(0)\n",
        "s3.upload_fileobj(buf, F1_MODELS_BUCKET, model_key_deg)\n",
        "buf_meta = io.BytesIO()\n",
        "pickle.dump(meta_deg, buf_meta)\n",
        "buf_meta.seek(0)\n",
        "s3.upload_fileobj(buf_meta, F1_MODELS_BUCKET, meta_key_deg)\n",
        "print('Uploaded degradation model to s3://%s/%s' % (F1_MODELS_BUCKET, model_key_deg))\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# 6. Prototype Pit Strategy Optimizer and Export Model/Rules\n",
        "class SimplePitStrategyModel:\n",
        "    def predict(self, X):\n",
        "        # X: [[current_lap, total_laps, tire_age_laps, gap_ahead, gap_behind]]\n",
        "        out = []\n",
        "        for (current_lap, total_laps, tire_age, gap_ahead, gap_behind) in X:\n",
        "            rel = current_lap / max(total_laps, 1)\n",
        "            if tire_age > 20 or rel > 0.7:\n",
        "                out.append(0.1)  # pit_now\n",
        "            elif gap_ahead < 2.0 and gap_behind > 4.0:\n",
        "                out.append(0.3)  # undercut opportunity\n",
        "            else:\n",
        "                out.append(0.8)  # stay_out\n",
        "        return out\n",
        "\n",
        "strategy_model = SimplePitStrategyModel()\n",
        "\n",
        "version_strat = 'v1_strategy'\n",
        "model_key_strat = f'strategy-models/{version_strat}/model.pkl'\n",
        "meta_key_strat = f'strategy-models/{version_strat}/metadata.pkl'\n",
        "meta_strat = {\n",
        "    'version': version_strat,\n",
        "    'description': 'Rule-based pit strategy with undercut detection',\n",
        "}\n",
        "buf = io.BytesIO()\n",
        "pickle.dump(strategy_model, buf)\n",
        "buf.seek(0)\n",
        "s3.upload_fileobj(buf, F1_MODELS_BUCKET, model_key_strat)\n",
        "buf_meta = io.BytesIO()\n",
        "pickle.dump(meta_strat, buf_meta)\n",
        "buf_meta.seek(0)\n",
        "s3.upload_fileobj(buf_meta, F1_MODELS_BUCKET, meta_key_strat)\n",
        "print('Uploaded strategy model to s3://%s/%s' % (F1_MODELS_BUCKET, model_key_strat))\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# 7. Design and Seed F1 Feature Store in DynamoDB\n",
        "from botocore.exceptions import ClientError\n",
        "\n",
        "def create_f1_features_table():\n",
        "    try:\n",
        "        table = dynamodb.create_table(\n",
        "            TableName=F1_FEATURES_TABLE,\n",
        "            KeySchema=[\n",
        "                {'AttributeName': 'feature_type', 'KeyType': 'HASH'},\n",
        "                {'AttributeName': 'feature_id', 'KeyType': 'RANGE'},\n",
        "            ],\n",
        "            AttributeDefinitions=[\n",
        "                {'AttributeName': 'feature_type', 'AttributeType': 'S'},\n",
        "                {'AttributeName': 'feature_id', 'AttributeType': 'S'},\n",
        "            ],\n",
        "            BillingMode='PAY_PER_REQUEST',\n",
        "        )\n",
        "        table.wait_until_exists()\n",
        "        print('Created table', F1_FEATURES_TABLE)\n",
        "    except ClientError as e:\n",
        "        if e.response['Error']['Code'] != 'ResourceInUseException':\n",
        "            raise\n",
        "        print('Table already exists')\n",
        "\n",
        "def seed_f1_features():\n",
        "    table = dynamodb.Table(F1_FEATURES_TABLE)\n",
        "    items = [\n",
        "        {\n",
        "            'feature_type': 'track',\n",
        "            'feature_id': 'monaco',\n",
        "            'track_name': 'Monaco',\n",
        "            'lap_distance_km': 3.337,\n",
        "            'corners': 19,\n",
        "            'avg_speed_kmh': 160,\n",
        "            'degradation_factor': 1.2,\n",
        "        },\n",
        "        {\n",
        "            'feature_type': 'track',\n",
        "            'feature_id': 'silverstone',\n",
        "            'track_name': 'Silverstone',\n",
        "            'lap_distance_km': 5.891,\n",
        "            'corners': 18,\n",
        "            'avg_speed_kmh': 230,\n",
        "            'degradation_factor': 0.9,\n",
        "        },\n",
        "        {\n",
        "            'feature_type': 'tire',\n",
        "            'feature_id': 'soft_c5',\n",
        "            'compound': 'SOFT',\n",
        "            'optimal_temp_range': [90, 110],\n",
        "            'expected_life_laps': 15,\n",
        "            'grip_level': 1.0,\n",
        "        },\n",
        "        {\n",
        "            'feature_type': 'weather',\n",
        "            'feature_id': 'current_bahrain',\n",
        "            'track_temp': 42,\n",
        "            'air_temp': 28,\n",
        "            'humidity': 45,\n",
        "            'conditions': 'dry',\n",
        "        },\n",
        "    ]\n",
        "    with table.batch_writer() as batch:\n",
        "        for item in items:\n",
        "            batch.put_item(Item=item)\n",
        "    print('Seeded F1 feature store with', len(items), 'items')\n",
        "\n",
        "create_f1_features_table()\n",
        "seed_f1_features()\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# 8–10. Lambda Inference Core, Model Loading/Caching, and Feature Access Layer\n",
        "from aws_xray_sdk.core import xray_recorder, patch_all as xray_patch_all\n",
        "xray_patch_all()\n",
        "\n",
        "import sys as _sys\n",
        "repo_root = pathlib.Path('lambda') / 'inference'\n",
        "_sys.path.insert(0, str(repo_root.parent))\n",
        "\n",
        "from inference.handler import lambda_handler\n",
        "\n",
        "sample_event = {\n",
        "    'httpMethod': 'POST',\n",
        "    'path': '/predict/laptime',\n",
        "    'body': json.dumps({\n",
        "        'driver': 'VER',\n",
        "        'track': 'monaco',\n",
        "        'tire_compound': 'SOFT',\n",
        "        'tire_age_laps': 5,\n",
        "        'fuel_load_kg': 60,\n",
        "        'track_temp': 38,\n",
        "        'air_temp': 24,\n",
        "    }),\n",
        "}\n",
        "# response = lambda_handler(sample_event, None)  # Requires deployed models in S3\n",
        "# print(response)\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# 11. AWS CDK Stack (Python) for API Gateway, Lambda, DynamoDB, S3\n",
        "cdk_stack_example = r'''\n",
        "from aws_cdk import (\n",
        "    Stack,\n",
        "    Duration,\n",
        "    aws_s3 as s3,\n",
        "    aws_dynamodb as dynamodb,\n",
        "    aws_lambda as _lambda,\n",
        "    aws_apigateway as apigw,\n",
        ")\n",
        "from constructs import Construct\n",
        "\n",
        "class F1InferenceStack(Stack):\n",
        "    def __init__(self, scope: Construct, construct_id: str, **kwargs) -> None:\n",
        "        super().__init__(scope, construct_id, **kwargs)\n",
        "\n",
        "        bucket = s3.Bucket(self, 'F1ModelsBucket',\n",
        "            bucket_name='f1-ml-models',\n",
        "            versioned=True,\n",
        "        )\n",
        "\n",
        "        features = dynamodb.Table(self, 'F1FeaturesTable',\n",
        "            table_name='f1-features',\n",
        "            partition_key={'name': 'feature_type', 'type': dynamodb.AttributeType.STRING},\n",
        "            sort_key={'name': 'feature_id', 'type': dynamodb.AttributeType.STRING},\n",
        "            billing_mode=dynamodb.BillingMode.PAY_PER_REQUEST,\n",
        "        )\n",
        "\n",
        "        fn = _lambda.Function(self, 'F1InferenceFn',\n",
        "            runtime=_lambda.Runtime.PYTHON_3_11,\n",
        "            handler='inference.handler.lambda_handler',\n",
        "            code=_lambda.Code.from_asset('../lambda'),\n",
        "            memory_size=256,\n",
        "            timeout=Duration.seconds(5),\n",
        "            environment={\n",
        "                'F1_MODELS_BUCKET': bucket.bucket_name,\n",
        "                'F1_FEATURES_TABLE': features.table_name,\n",
        "            },\n",
        "        )\n",
        "        bucket.grant_read(fn)\n",
        "        features.grant_read_write_data(fn)\n",
        "\n",
        "        api = apigw.RestApi(self, 'F1InferenceApi',\n",
        "            rest_api_name='F1 Race Strategy',\n",
        "        )\n",
        "        predict = api.root.add_resource('predict')\n",
        "        for path in ['laptime','pit-strategy','tire-degradation']:\n",
        "            res = predict.add_resource(path)\n",
        "            res.add_method('POST', apigw.LambdaIntegration(fn))\n",
        "'''\n",
        "print(cdk_stack_example)\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# 12. CloudWatch Metrics, Alarms, and Dashboard JSON\n",
        "dashboard = {\n",
        "  'widgets': [\n",
        "    {\n",
        "      'type': 'metric',\n",
        "      'x': 0, 'y': 0, 'width': 12, 'height': 6,\n",
        "      'properties': {\n",
        "        'title': 'Lap Time Prediction Error by Track',\n",
        "        'metrics': [\n",
        "          ['F1Inference', 'prediction_error', 'track', 'monaco'],\n",
        "          ['.', 'prediction_error', 'track', 'silverstone'],\n",
        "          ['.', 'prediction_error', 'track', 'spa'],\n",
        "        ],\n",
        "        'stat': 'Average',\n",
        "        'period': 60,\n",
        "      }\n",
        "    },\n",
        "  ]\n",
        "}\n",
        "print(json.dumps(dashboard, indent=2))\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# 13. Race Simulation Load Test (20-Car Scenario)\n",
        "import asyncio, time as _time\n",
        "import httpx\n",
        "\n",
        "API_BASE = os.environ.get('F1_API_BASE', 'https://your-api-id.execute-api.region.amazonaws.com/prod')\n",
        "\n",
        "async def simulate_driver(client, driver_code: str, total_laps: int = 58):\n",
        "    latencies = []\n",
        "    for lap in range(1, total_laps + 1):\n",
        "        payload = {\n",
        "            'driver': driver_code,\n",
        "            'track': 'monaco',\n",
        "            'tire_compound': 'SOFT',\n",
        "            'tire_age_laps': max(1, lap - 1),\n",
        "            'fuel_load_kg': max(5, 100 - lap),\n",
        "            'track_temp': 38,\n",
        "            'air_temp': 24,\n",
        "        }\n",
        "        t0 = _time.perf_counter()\n",
        "        r = await client.post(f'{API_BASE}/predict/laptime', json=payload)\n",
        "        latencies.append((_time.perf_counter() - t0) * 1000.0)\n",
        "    return latencies\n",
        "\n",
        "async def run_race():\n",
        "    async with httpx.AsyncClient(timeout=5.0) as client:\n",
        "        tasks = [simulate_driver(client, f'DRV{i:02d}') for i in range(1, 21)]\n",
        "        all_latencies = await asyncio.gather(*tasks)\n",
        "    flat = [x for sub in all_latencies for x in sub]\n",
        "    flat.sort()\n",
        "    p95 = flat[int(0.95 * len(flat))] if flat else None\n",
        "    print('p95 latency (ms):', p95)\n",
        "\n",
        "# asyncio.run(run_race())  # Enable when API is deployed\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# 14. Prediction Accuracy Validation Against 2024 Races\n",
        "def validate_against_2024(api_base: str, year: int = 2024, race: str = 'Bahrain'):\n",
        "    session = fastf1.get_session(year, race, 'R')\n",
        "    session.load()\n",
        "    laps = session.laps.reset_index(drop=True)\n",
        "    import requests\n",
        "    errors = []\n",
        "    for _, row in laps.iterrows():\n",
        "        if pd.isna(row['LapTime']):\n",
        "            continue\n",
        "        payload = {\n",
        "            'driver': row['Driver'],\n",
        "            'track': race.lower(),\n",
        "            'tire_compound': str(row['Compound']),\n",
        "            'tire_age_laps': int(row['TyreLife'] or 0),\n",
        "            'fuel_load_kg': 60,\n",
        "            'track_temp': 38,\n",
        "            'air_temp': 24,\n",
        "        }\n",
        "        r = requests.post(f'{api_base}/predict/laptime', json=payload)\n",
        "        pred = r.json()['predicted_laptime']\n",
        "        mins, secs = pred.split(':')\n",
        "        pred_secs = int(mins) * 60 + float(secs)\n",
        "        actual_secs = row['LapTime'].total_seconds()\n",
        "        errors.append(abs(pred_secs - actual_secs))\n",
        "    arr = np.array(errors)\n",
        "    return {\n",
        "        'rmse': float(np.sqrt((arr ** 2).mean())),\n",
        "        'mae': float(np.abs(arr).mean()),\n",
        "    }\n",
        "\n",
        "# metrics = validate_against_2024(API_BASE)\n",
        "# metrics\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# 15. F1-Branded Architecture Diagram with graphviz\n",
        "from graphviz import Digraph\n",
        "\n",
        "dot = Digraph('F1Inference', format='png')\n",
        "dot.attr(rankdir='LR')\n",
        "dot.node('dash', 'Race Engineer\\nDashboard (Pit Wall)', shape='box', style='filled', color='lightgrey')\n",
        "dot.node('api', 'API Gateway', shape='box')\n",
        "dot.node('lambda', 'Lambda\\nF1 Inference', shape='box')\n",
        "dot.node('ddb', 'DynamoDB\\nF1 Feature Store', shape='cylinder')\n",
        "dot.node('s3', 'S3\\nF1 Models', shape='folder')\n",
        "dot.node('cw', 'CloudWatch / X-Ray', shape='ellipse')\n",
        "dot.edges([('dash','api'), ('api','lambda')])\n",
        "dot.edges([('lambda','ddb'), ('lambda','s3'), ('lambda','cw')])\n",
        "dot.render('f1_inference_architecture', cleanup=True)\n",
        "dot\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {"language": "python"},
      "source": [
        "# 16. Example API Calls for Monaco, Silverstone, and Spa\n",
        "import requests\n",
        "\n",
        "def call_example(track: str, tire: str, tire_age: int, fuel: int):\n",
        "    payload = {\n",
        "        'driver': 'VER',\n",
        "        'track': track,\n",
        "        'tire_compound': tire,\n",
        "        'tire_age_laps': tire_age,\n",
        "        'fuel_load_kg': fuel,\n",
        "        'track_temp': 38,\n",
        "        'air_temp': 24,\n",
        "    }\n",
        "    r = requests.post(f'{API_BASE}/predict/laptime', json=payload)\n",
        "    print(track, r.json())\n",
        "\n",
        "# call_example('monaco', 'SOFT', 5, 60)\n",
        "# call_example('silverstone', 'MEDIUM', 15, 40)\n",
        "# call_example('spa', 'HARD', 22, 30)\n"
      ]
    }
  ]
}


{'cells': [{'cell_type': 'markdown',
   'metadata': {'language': 'markdown'},
   'source': ['# F1 Race Strategy ML Pipeline (Serverless AWS)\n',
    '\n',
    'End-to-end notebook to collect FastF1 telemetry, train lap time / tire degradation / pit strategy models, and deploy them to an AWS serverless inference stack with Lambda, API Gateway, DynamoDB, and S3.']},
  {'cell_type': 'code',
   'metadata': {'language': 'python'},
   'source': ['# 1. Set Up F1 ML Environment and Dependencies\n',
    '!pip install --quiet fastf1 pandas numpy scikit-learn xgboost boto3 awscli aws-xray-sdk graphviz httpx matplotlib\n',
    '\n',
    'import os, pathlib, datetime, json, math, random\n',
    'import boto3\n',
    '\n',
    "AWS_REGION = os.environ.get('AWS_REGION', 'eu-west-1')\n",
    "AWS_PROFILE = os.environ.get('AWS_PROFILE', 'default')\n",
    "F1_MODELS_BUCKET = os.environ.get('F1_MODELS_BUCKET', 'f1-ml-models')\n",
    "F1_FEATURES_TABLE = os.environ.get('F1_FEATURES_TABLE', 'f1-features'

Searched codebase for "# Predicting Formula 1 Race Strategy with Machine Learning (Serverless AWS)

This project is a production-grade ML inference system for live Formula 1 race strategy: lap time prediction, tire degradation estimation, and pit stop recommendations. It is built on AWS serverless primitives (API Gateway, Lambda, DynamoDB, S3) to showcase scalable ML infrastructure with a memorable F1 use case.

## High-Level Architecture

- Race Engineer Dashboard (pit wall UI)
- Amazon API Gateway (REST API)
- AWS Lambda (F1 inference engine)
- Amazon DynamoDB (F1 feature store: tracks, tires, weather)
- Amazon S3 (versioned ML models for lap time, degradation, strategy)
- Amazon CloudWatch + AWS X-Ray (monitoring, tracing)

An F1-themed architecture diagram is generated via Graphviz in the notebook: see `data/f1_training_pipeline.ipynb` section 15.

## Repository Layout

- `lambda/inference/`
  - `handler.py` – Lambda entrypoint exposing:
    - `POST /predict/laptime`
    - `POST /predict/pit-strategy`
    - `POST /predict/tire-degradation`
  - `models.py` – S3 model loading, caching, X-Ray instrumentation
  - `features.py` – DynamoDB feature store access helpers
  - `requirements.txt` – Lambda dependencies
- `data/`
  - `collect_f1_data.py` – FastF1 telemetry collection for 2023–2024
  - `train_laptime_model.py` – train & persist lap time regressor
  - `train_strategy_model.py` – strategy score model
  - `train_degradation_model.py` – tire degradation model
  - `f1_training_pipeline.ipynb` – end-to-end training & deployment notebook
- `infra/`
  - `app.py` – CDK app entrypoint
  - `f1_inference_stack.py` – CDK stack (S3, DynamoDB, Lambda, API Gateway, dashboard)
  - `f1_dashboard.json` – CloudWatch dashboard definition
- `scripts/`
  - `populate_f1_features.py` – seed DynamoDB with track/tire/weather
  - `deploy_model.py` – upload local models to S3
- `tests/`
  - `load_test_race_simulation.py` – 20-car race load test
  - `validate_predictions.py` – compare predictions vs real 2024 race data

## Endpoints

### POST /predict/laptime

Example request (Monaco, Verstappen quali-style lap):



In [ ]:
{
  "driver": "VER",
  "track": "monaco",
  "tire_compound": "SOFT",
  "tire_age_laps": 5,
  "fuel_load_kg": 60,
  "track_temp": 38,
  "air_temp": 24
}



Example response shape:



In [ ]:
{
  "predicted_laptime": "1:14.532",
  "confidence_interval": ["1:14.200", "1:14.900"],
  "model_version": "v2_2024_season",
  "latency_ms": 87.3,
  "track": "Monaco",
  "conditions": "optimal"
}



### POST /predict/pit-strategy



In [ ]:
{
  "current_lap": 25,
  "total_laps": 58,
  "current_position": 3,
  "tire_compound": "MEDIUM",
  "tire_age_laps": 18,
  "gap_ahead_seconds": 3.2,
  "gap_behind_seconds": 5.8,
  "track": "silverstone"
}



### POST /predict/tire-degradation



In [ ]:
{
  "tire_compound": "HARD",
  "current_lap": 35,
  "laps_on_tire": 22,
  "track": "spa",
  "track_temp": 28,
  "driver_style": "aggressive"
}



## Deployment Guide (AWS CDK)

Prereqs:
- AWS account & CLI configured
- Python 3.11
- AWS CDK (v2) installed

Setup:



In [ ]:
cd infra
pip install -r ../lambda/inference/requirements.txt
pip install "aws-cdk-lib" "constructs"
cdk bootstrap
cdk deploy



After deploy, note the `F1ApiUrl` output (base API URL).

### Seed Feature Store



In [ ]:
export AWS_REGION=eu-west-1
python scripts/populate_f1_features.py



### Train and Deploy Models



In [ ]:
# Collect telemetry and engineer features
python data/collect_f1_data.py

# Train models
python data/train_laptime_model.py
python data/train_strategy_model.py
python data/train_degradation_model.py

# Upload to S3 in versioned layout
python scripts/deploy_model.py



## Load Testing: 20-Car Race Simulation

The script `tests/load_test_race_simulation.py` simulates 20 concurrent drivers over a 58-lap race and reports mean, p95, and p99 latency. Set `F1_API_BASE` to your API base URL and run:



In [ ]:
export F1_API_BASE="https://your-api-id.execute-api.region.amazonaws.com/prod"
python tests/load_test_race_simulation.py



Target metrics:
- p95 latency < 120 ms
- throughput ~150 RPS

## Validation vs Real F1 2024

`tests/validate_predictions.py` replays a real 2024 race via FastF1, hits the `/predict/laptime` endpoint per lap, and computes RMSE/MAE between predicted and actual lap times.



In [ ]:
export F1_API_BASE="https://your-api-id.execute-api.region.amazonaws.com/prod"
python tests/validate_predictions.py



Use the printed metrics directly as resume bullets (e.g. "<0.4s RMSE on Bahrain 2024").

## CloudWatch Monitoring

The CDK stack and `infra/f1_dashboard.json` configure an F1-specific dashboard and alarms:
- Lap time prediction error by track
- Inference latency p95 by prediction type
- Requests by track and model version
- Alarms for high latency, error rate, and throttling

You can import `infra/f1_dashboard.json` via the AWS Console or AWS CLI to create the dashboard.

## Example Demo Snippet (Monaco, Silverstone, Spa)



In [4]:
import requests

API_BASE = "https://your-api-id.execute-api.region.amazonaws.com/prod"

for track, tire, age, fuel in [
    ("monaco", "SOFT", 5, 60),
    ("silverstone", "MEDIUM", 15, 40),
    ("spa", "HARD", 22, 30),
]:
    resp = requests.post(
        f"{API_BASE}/predict/laptime",
        json={
            "driver": "VER",
            "track": track,
            "tire_compound": tire,
            "tire_age_laps": age,
            "fuel_load_kg": fuel,
            "track_temp": 38,
            "air_temp": 24,
        },
    )
    print(track, resp.json())

ConnectionError: HTTPSConnectionPool(host='your-api-id.execute-api.region.amazonaws.com', port=443): Max retries exceeded with url: /prod/predict/laptime (Caused by NameResolutionError("HTTPSConnection(host='your-api-id.execute-api.region.amazonaws.com', port=443): Failed to resolve 'your-api-id.execute-api.region.amazonaws.com' ([Errno 11001] getaddrinfo failed)"))



## Suggested Resume Bullets

- Built a production-style AWS serverless ML inference system for Formula 1 race strategy predictions (lap time, tire degradation, pit windows), handling ~150 RPS with p95 latency targets under 120 ms.
- Deployed three versioned ML models (lap time, tire degradation, pit strategy) to S3 and served them via Lambda with model caching, X-Ray tracing, and canary-ready configuration using environment-driven model versions.
- Implemented a real-time F1 feature store on DynamoDB and S3 for track conditions, tire characteristics, and weather data, enabling low-latency feature fetches for live race strategy.
- Built CloudWatch dashboards and alarms tracking F1-specific metrics (prediction error vs actual 2024 results, latency percentiles, per-track traffic, and model version traffic split) with automated alerting for model or system degradation.
gain some context about this project from the file sturctue so you can help me code it 
resides in https://github.com/kolinatasha/f1-racing-ml-inference", 15 results

Let me read the key files across the project to understand the current state.   

Read [](file:///c%3A/Users/mithu/OneDrive/Desktop/natasha/f1-racing-ml-inference/lambda/inference/handler.py#1-1), lines 1 to 100

Read [](file:///c%3A/Users/mithu/OneDrive/Desktop/natasha/f1-racing-ml-inference/lambda/inference/models.py#1-1), lines 1 to 100

Read [](file:///c%3A/Users/mithu/OneDrive/Desktop/natasha/f1-racing-ml-inference/lambda/inference/features.py)

Read [](file:///c%3A/Users/mithu/OneDrive/Desktop/natasha/f1-racing-ml-inference/lambda/inference/handler.py#100-100), lines 100 to 205

Read [](file:///c%3A/Users/mithu/OneDrive/Desktop/natasha/f1-racing-ml-inference/infra/f1_inference_stack.py#1-1), lines 1 to 150

Read [](file:///c%3A/Users/mithu/OneDrive/Desktop/natasha/f1-racing-ml-inference/data/collect_f1_data.py#1-1), lines 1 to 80

Read [](file:///c%3A/Users/mithu/OneDrive/Desktop/natasha/f1-racing-ml-inference/data/train_laptime_model.py#1-1), lines 1 to 100

Read [](file:///c%3A/Users/mithu/OneDrive/Desktop/natasha/f1-racing-ml-inference/scripts/populate_f1_features.py#1-1), lines 1 to 100

Read [](file:///c%3A/Users/mithu/OneDrive/Desktop/natasha/f1-racing-ml-inference/tests/load_test_race_simulation.py)

Read [](file:///c%3A/Users/mithu/OneDrive/Desktop/natasha/f1-racing-ml-inference/scripts/deploy_model.py)

Read [](file:///c%3A/Users/mithu/OneDrive/Desktop/natasha/f1-racing-ml-inference/tests/validate_predictions.py)

Read [](file:///c%3A/Users/mithu/OneDrive/Desktop/natasha/f1-racing-ml-inference/lambda/inference/requirements.txt)

I've read all the key files. Here's what the codebase looks like and the issues that need attention before it runs end-to-end:

---

## Project Overview

**Stack:** FastF1 telemetry → scikit-learn/XGBoost training → S3 model storage → Lambda inference → API Gateway REST API → CDK-deployed AWS infra.

### What's fully implemented
| File | Status |
|---|---|
| handler.py | ✅ All 3 routes wired, X-Ray, formatting |
| models.py | ✅ S3 load + in-process cache + X-Ray |
| features.py | ✅ DynamoDB helpers for track/tire/weather |
| f1_inference_stack.py | ✅ CDK stack: S3, DynamoDB (2 tables), Lambda, API GW, CW alarms |
| deploy_model.py | ✅ Uploads models tree to S3 |
| load_test_race_simulation.py | ✅ 20-driver async httpx load test |
| validate_predictions.py | ✅ FastF1 replay vs. API RMSE/MAE |
| train_laptime_model.py | ✅ RF + XGBoost, best-model selection, pickle save |

### Known bugs / inconsistencies to fix

1. **Feature mismatch (critical)** — Training uses `["tire_life_laps", "fuel_load_proxy", "tire_degradation_feature"]` (train_laptime_model.py), but the Lambda handler builds `["tire_age_laps", "track_temp", "air_temp", "fuel_load_kg", "track_degradation_factor", "tire_grip_level"]` (handler.py). The model will receive the wrong feature count/order at inference time.

2. **DynamoDB tire key mismatch** — features.py looks up tires by `compound.lower()` (e.g. `"soft"`), but populate_f1_features.py seeds them with `feature_id = "soft_c5"`, `"medium_c3"`, `"hard_c1"`. Tire lookups will always miss.

3. **Missing imports in populate script** — populate_f1_features.py uses `List` and `Dict` in `seed_items()` without importing them (`from typing import List, Dict`). Script will crash on startup.

4. **collect_f1_data.py has no `main()`** — The file defines `collect_season_laps()` and `feature_engineering_laptime()` but the bottom of the file (lines 80–103) is not shown; worth confirming it actually writes `processed/laps_features.parquet`.

5. **train_laptime_model.py trains on only 3 features** but those 3 don't include `track_temp`, `air_temp`, `fuel_load_kg` that the handler passes — all three training scripts need their `FEATURE_COLUMNS` aligned with the handler's feature vectors.

---

Tell me what you'd like to work on and I can jump straight into it — e.g. aligning features end-to-end, fixing the DynamoDB key scheme, completing the other two training scripts, or anything else.